In [2]:
import pickle
from itertools import product

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector, Parameter
from qiskit.qasm3 import dump as qasm3_dump

from hubo_qaoa.utils.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian
from hubo_qaoa.utils.gfa_utils import gfa_file_to_graph
from hubo_qaoa.utils.parameterise_circuit import parameterise_circuit
from hubo_qaoa.utils.lr_qaoa import get_LR_qaoa_circuit

In [3]:
data_file = '/lustre/scratch127/qpg/jc59/new_hubo_formulation/circuit_depths/results.couplingall.precompute.0.pkl'
with open(data_file, 'rb') as f:
    res = pickle.load(f)

In [4]:
res.keys()

dict_keys(['test_N2_W2', 'trivial', 'test_N3_W4', 'test_N4_W5', 'test_N7_W2', 'test_N7_W3', 'test_N7_W4', 'test_N8_W2', 'test_N8_W3', 'test_N8_W5', 'test_N8_W6'])

In [5]:
res['test_N2_W2']['rzz']

{'count': 10,
 'depth': 9,
 'layers': 0,
 'layout': Layout({
 0: <Qubit register=(4, "q"), index=0>,
 1: <Qubit register=(4, "q"), index=1>,
 2: <Qubit register=(4, "q"), index=2>,
 3: <Qubit register=(4, "q"), index=3>
 }),
 'circuit': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x7f056bb562c0>}

In [6]:
data_file = '/lustre/scratch127/qpg/jc59/new_hubo_formulation/circuit_depths/results.couplingall.precompute.0.pkl'
with open(data_file, 'rb') as f:
    res = pickle.load(f)
    
delta_b_fixed = 0.75
delta_g_fixed = 0.30
for (filename, copy_numbers, name) in zip(
        [
            'test_N2_W2',
            'trivial',
            'test_N3_W4',
            'test_N7_W2',
            'test_N4_W5'
        ],
        [
            [1,1],
            [1,1,1],
            [2,1,1],
            [1,0,0,0,0,0,1],
            [2,1,1,1]
        ],
        [
            'I4',
            'I9',
            'I12',
            'I8',
            'I15'
        ]
    ):
    filepath = f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/{filename}.gfa'
    graph, n, V, T = gfa_file_to_graph(filepath, copy_numbers)
    hamiltonian = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)
    with open(f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/new_hubo_formulation/ionq/hubo_hamiltonian_{name}.pkl', 'wb') as f:
        pickle.dump(hamiltonian, f)

    cost_circuit = parameterise_circuit(res[filename]['rzz']['circuit'], parameter=Parameter('γ'))
    num_qubits: int = cost_circuit.num_qubits    
    print(filename, name, num_qubits)

    phis = ParameterVector('ϕ', num_qubits)
    for p in [1,2,3,4,5]:
        fixed_qc, circuit = get_LR_qaoa_circuit(p, delta_b_fixed, delta_g_fixed, num_qubits, cost_circuit, None, phis, True)
        with open(f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/new_hubo_formulation/ionq/qasm3/iter_qaoa_hubo_circuit_{name}_{p}.txt', 'w') as f:
            qasm3_dump(fixed_qc, f)

Keeping constraints at times: [0]
test_N2_W2 I4 4
11:35:29 - hubo_qaoa.utils.lr_qaoa - INFO - p = 1. Circuit depth: 15
11:35:29 - hubo_qaoa.utils.lr_qaoa - INFO - p = 2. Circuit depth: 28
11:35:29 - hubo_qaoa.utils.lr_qaoa - INFO - p = 3. Circuit depth: 41
11:35:29 - hubo_qaoa.utils.lr_qaoa - INFO - p = 4. Circuit depth: 54
11:35:29 - hubo_qaoa.utils.lr_qaoa - INFO - p = 5. Circuit depth: 67
Keeping constraints at times: [0 1]
trivial I9 9
11:35:29 - hubo_qaoa.utils.lr_qaoa - INFO - p = 1. Circuit depth: 121
11:35:29 - hubo_qaoa.utils.lr_qaoa - INFO - p = 2. Circuit depth: 240
11:35:29 - hubo_qaoa.utils.lr_qaoa - INFO - p = 3. Circuit depth: 359
11:35:29 - hubo_qaoa.utils.lr_qaoa - INFO - p = 4. Circuit depth: 478
11:35:29 - hubo_qaoa.utils.lr_qaoa - INFO - p = 5. Circuit depth: 597
Keeping constraints at times: [1 2 0]
test_N3_W4 I12 12
11:35:30 - hubo_qaoa.utils.lr_qaoa - INFO - p = 1. Circuit depth: 162
11:35:30 - hubo_qaoa.utils.lr_qaoa - INFO - p = 2. Circuit depth: 320
11:35:30 -